# Lab 61: The Sentiment Reader

*Brain Lab: 84 Experiments to Decode Your Biological and Artificial Intelligence* · Chapter 7, *The Decision Bias*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/labphonlab/brain-lab-support/blob/main/notebooks/lab-61.ipynb)

**What you will do:** score how well a listener identifies emotions from your recordings, compare their accuracy with chance level, pool the class's cross-language results, and see how a pretrained speech-emotion model classifies the same clips.

**How to use.** Run the cells from top to bottom (Shift+Enter). Nothing needs installing, and nothing you do here can break anything: *Runtime → Restart and run all* returns it to the start. To keep your own results, use *File → Save a copy in Drive*.

In [ ]:
# Setup: run this cell first (Shift+Enter).
import platform, numpy as np, pandas as pd, matplotlib
import matplotlib.pyplot as plt
np.random.seed(2026)
print("Python", platform.python_version(), "| numpy", np.__version__, "| pandas", pd.__version__)

## The experiment in the book

In Lab 61 you recorded a friend saying the neutral sentence "I will be there at five o'clock" in
four emotional tones — happy, angry, sad, and sarcastic — then played the clips to a third
person and asked them to identify the emotion from tone alone. Most people succeed with high
accuracy, even though the words never change; the emotional information is carried by
**prosody** — melody, rhythm, and timbre. The book notes that basic emotions are recognised
above chance even by listeners who do not speak the language being spoken (Pell et al., 2009),
though cultural "display rules" mean cross-language recognition is above chance, not perfect.

## Record your results

Enter your listener's guess for each of the four clips.

In [ ]:
results = pd.DataFrame({
    "clip": ["happy", "angry", "sad", "sarcastic"],
    "listener_guess": ["happy", "angry", "sarcastic", "sarcastic"],  # example data: replace with yours
})
results["correct"] = results.clip == results.listener_guess
display(results)
print(f"Listener accuracy: {results.correct.mean():.0%} ({results.correct.sum()}/{len(results)})")

## Compare

The lab does not give a single published accuracy figure for this exact task, but with four
emotion categories, chance-level guessing succeeds 1 in 4 times (25%). The lab's claim is that
listeners do much better than this, even across languages they do not speak.

In [ ]:
chance = 1 / 4
acc = results.correct.mean()
print(f"Your listener's accuracy: {acc:.0%}   |   chance level with 4 categories: {chance:.0%}")
if acc > chance:
    print("Your listener beat chance, consistent with the lab's claim that prosody carries")
    print("real emotional information even when the words are identical.")
else:
    print("Your listener did not clearly beat chance with only 4 clips -- a single trial per")
    print("emotion is a very small sample; the published effect is about averages across many")
    print("listeners and many clips, not a guarantee for any one person.")

## The AI side

A pretrained speech-emotion-recognition (SER) model can classify the acoustic signal directly,
the way the book's "AI Connection" section describes. Upload your happy, angry, and sad clips
(skip sarcastic -- the model below has no such category) and see whether its top guess matches
the emotion you intended.

*This part downloads a small open model or library, so it runs in Colab (it may take a minute the first time).*

In [ ]:
from transformers import pipeline

try:
    from google.colab import files
    print("Upload your recordings, named happy.wav, angry.wav, and sad.wav.")
    uploaded = list(files.upload().keys())
except ImportError:
    uploaded = []  # not running in Colab

clf = pipeline("audio-classification", model="superb/wav2vec2-base-superb-er")
rows = []
for fname in uploaded:
    intended = fname.rsplit(".", 1)[0].lower()
    top = clf(fname, top_k=4)[0]
    rows.append({"file": fname, "intended emotion": intended,
                 "model's top label": top["label"], "confidence": round(top["score"], 2)})
ai_df = pd.DataFrame(rows)
display(ai_df)

This model was trained on IEMOCAP, an acted-speech dataset with only four categories (neutral,
happy, angry, sad), so it cannot represent sarcasm at all -- a limitation that echoes the
book's point that sarcasm requires detecting a *mismatch* between literal meaning and prosody,
not just classifying prosody on its own. The model may also perform worse on a phone
recording of an amateur actor than on the clean, professional recordings it was trained on.
Treat any mismatch between the model's guess and your intended emotion as a reminder that
speech-emotion recognition, like the small text and vision models elsewhere in this book, is
imperfect and dataset-dependent, not a definitive judge of what emotion was "really" there.

## Pool the class data

Pool listener accuracy across the class, separating clips the listener heard in a language
they speak from clips in a language they do not (anonymous IDs, not names), to explore Pell et
al.'s (2009) cross-language claim.

In [ ]:
import io
csv_text = """id,language_familiarity,correct,total
P01,familiar,4,4
P01,unfamiliar,2,4
P02,familiar,3,4
P02,unfamiliar,3,4
P03,familiar,4,4
P03,unfamiliar,1,4
"""  # example data: replace with your class CSV, e.g. pd.read_csv("prosody_class.csv")
df = pd.read_csv(io.StringIO(csv_text))
df["accuracy"] = df.correct / df.total
summary = df.groupby("language_familiarity").accuracy.mean()
display(summary)
boot = {g: [df.loc[df.language_familiarity == g].accuracy.sample(frac=1, replace=True).mean()
            for _ in range(3000)] for g in df.language_familiarity.unique()}
for g, b in boot.items():
    lo, hi = np.percentile(b, [2.5, 97.5])
    print(f"{g}: mean accuracy {summary[g]:.0%} (95% bootstrap interval {lo:.0%}-{hi:.0%})")
print("Chance level with 4 categories: 25%. Pell et al. (2009) report above-chance recognition")
print("even for unfamiliar languages, but not as high as for a familiar one.")

## Questions to think about

1. Which of the four emotions was hardest for your listener to identify correctly? Does that match your own intuition about which emotions sound most similar in tone?
2. The book says cross-language recognition is "above chance, not perfect." Did the class's pooled unfamiliar-language accuracy clear the 25% chance level, and how far below the familiar-language accuracy did it fall?
3. The SER model has no sarcasm category. What acoustic or contextual information would a model need to add to detect sarcasm, beyond classifying prosody alone?
4. The book describes aprosodia -- a condition where patients understand words but not tone. If you covered the audio and only read a transcript of "I will be there at five o'clock," could you tell which of the four emotions was intended? What does that tell you about how much of the information lives in the words versus the tone?

## Challenge

Record the same sentence in a fifth emotional tone not tested in the lab (for example, bored or
fearful) and see whether your listener can identify it without being told the list of options in
advance (a harder, open-set version of the task).

In [ ]:
# Example data: replace with your own open-set trial.
intended_emotion = "bored"
listener_free_response = "tired"   # example: what the listener said, unprompted

print(f"Intended: {intended_emotion} | Listener said: {listener_free_response}")
print("An open-set response that is close in meaning (like 'tired' for 'bored') suggests the")
print("prosodic cue was picked up even without an exact label match. Replace the values above")
print("with your own recording and response.")

## Record what you ran

In [ ]:
NOTEBOOK_ID = "lab-61.ipynb"
# Reproducibility record: paste this output into your notes or report.
import sys, platform, datetime, numpy, pandas, matplotlib
print("Notebook:", NOTEBOOK_ID)
print("Run at  :", datetime.datetime.now().isoformat(timespec="seconds"))
print("Python  :", sys.version.split()[0], "on", platform.platform())
print("numpy", numpy.__version__, "| pandas", pandas.__version__, "| matplotlib", matplotlib.__version__)
print("Seed    : 2026")